In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, glob
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/"
LOOKUP_PATH = os.path.join("/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/taxi_zone_lookup_grid.csv")

DATA_DIR: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/
LOOKUP: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/taxi_zone_lookup_grid.csv True


In [3]:
file_list = sorted(glob.glob(os.path.join(DATA_DIR, "*_volume.csv")))
df_list = [pd.read_csv(f) for f in file_list]
df_all = pd.concat(df_list, ignore_index=True)

df_all = df_all.rename(columns={"locationid": "LocationID"})

df_lookup = pd.read_csv(LOOKUP_PATH)
df_merge = pd.merge(df_all, df_lookup, on="LocationID", how="left")

In [4]:
print("Merged rows:", len(df_merge))
print(df_merge.head())

Merged rows: 3030272
              time_bin  LocationID  start_volume  end_volume  Grid_X  Grid_Y
0  2001-01-05 11:30:00          71             1           0       5       6
1  2001-01-05 11:30:00          89             0           1       5       6
2  2002-12-31 23:00:00          48             1           0       4      12
3  2002-12-31 23:00:00          68             0           1       4      11
4  2002-12-31 23:00:00          79             1           0       4      10


In [5]:
df_merge["time_bin"] = pd.to_datetime(df_merge["time_bin"], errors="coerce")
df_merge = df_merge.dropna(subset=["time_bin", "Grid_X", "Grid_Y"])

df_merge["Grid_X"] = df_merge["Grid_X"].astype(int)
df_merge["Grid_Y"] = df_merge["Grid_Y"].astype(int)
df_merge["start_volume"] = pd.to_numeric(df_merge["start_volume"], errors="coerce").fillna(0).astype(np.float32)
df_merge["end_volume"] = pd.to_numeric(df_merge["end_volume"], errors="coerce").fillna(0).astype(np.float32)

In [6]:
df_merge = df_merge[(df_merge["time_bin"] >= "2018-01-01") & (df_merge["time_bin"] < "2019-01-01")].copy()

print("After cleaning:", len(df_merge))
print(df_merge[["time_bin","LocationID","Grid_X","Grid_Y","start_volume","end_volume"]].head())

After cleaning: 3028370
      time_bin  LocationID  Grid_X  Grid_Y  start_volume  end_volume
319 2018-01-01           3       7      18           0.0         2.0
320 2018-01-01           4       4      10          39.0        56.0
321 2018-01-01           7       6      12           9.0        45.0
322 2018-01-01           9       8      12           0.0         2.0
323 2018-01-01          10       8       8           0.0         5.0


In [7]:
df_merge = df_merge.sort_values("time_bin")

unique_times = sorted(df_merge["time_bin"].unique())
time_to_idx = {t:i for i,t in enumerate(unique_times)}
df_merge["time_idx"] = df_merge["time_bin"].map(time_to_idx).astype(int)

T = len(unique_times)
H = int(df_merge["Grid_Y"].max() + 1)
W = int(df_merge["Grid_X"].max() + 1)
C = 3

ys = df_merge["Grid_Y"].values
xs = df_merge["Grid_X"].values
ts = df_merge["time_idx"].values

start_vals = df_merge["start_volume"].values
end_vals = df_merge["end_volume"].values

data = np.zeros((T, C, H, W), dtype=np.float32)

data[ts, 0, ys, xs] = start_vals
data[ts, 1, ys, xs] = end_vals
data[ts, 2, ys, xs] = start_vals - end_vals

print("Nonzero:", np.count_nonzero(data))

Nonzero: 2628977


In [9]:
OUT_PATH = os.path.join(DATA_DIR, "taxi_volume_4d_tensor.npy")
np.save(OUT_PATH, data)

print("Saved:", OUT_PATH)

Saved: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/taxi_volume_4d_tensor.npy
